# 🧠 Core Idea (Don’t Forget This)

Use the right tool for the right job:

- 🧠 Agent → thinking, deciding, using tools
- ⚙️ Structured LLM → extracting + formatting

---

# ⚡ Architecture (80/20)

Image → Structured LLM → Agent → Structured LLM

That’s it.

---

# 🎯 Why This Works

- Structured LLM = reliable, predictable
- Agent = flexible, smart

👉 Mixing them correctly = production-ready system

---

# ❌ Common Mistake

Trying to make ONE agent do everything:

- extraction ❌
- reasoning ❌
- formatting ❌

→ becomes unstable and hard to debug

---

# ✅ Correct Pattern

Split responsibilities:

1. Extract data (structured)
2. Think + decide (agent)
3. Format output (structured)

---

# 🧠 Mental Model

- Agent = brain  
- Structured LLM = worker  

---

# 🔥 If You Only Remember One Thing

👉 Use agents for **decisions**  
👉 Use structured output for **data**

### This is an exercise that combines the following content
1. Memory
2. Tools (especially web search tool)
3. Multi-modal message

### Load `environment` variables 🌎

In [14]:
from dotenv import load_dotenv

load_dotenv()

True

### Define tavily web `search tool` 🔍

In [15]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)


### Create an agent equiped with web search `tool` ⚙️ (tavily under the hood) and with `memory` 🧠

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    checkpointer=InMemorySaver(),
)

### Get the image input 🏞️

In [17]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

### Convert image to base64 encoding 🔄

In [18]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

### Create the multi-modal message (including images and human messages)

In [20]:
from langchain.messages import HumanMessage

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Tell me what are there in the fridge"},
        {"type": "image", "base64": img_b64, "mime_type": "image/png"},
    ]
)

# ! This is very important, without this the memory won't work
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [multimodal_question]}, config)

print(response["messages"][-1].content)

Here’s what I can see in the fridge, from top to bottom:

- Top shelf
  - Olive oil bottle
  - Half a lime
  - Cucumber
  - Celery stalks
  - Green apple

- Middle shelf
  - Beet greens (with red stems)
  - Strawberries
  - Beet (root)
  - Bottle of red juice (likely tomato juice)

- Bottom shelf
  - Carrots with greens
  - Two bottles of orange juice
  - Citrus: one orange cut in half, plus two whole oranges
  - Two lemons

Want me to suggest a quick snack or a recipe using these ingredients?


### Test the memory 🧠

In [21]:
from pprint import pprint

question = HumanMessage(
    content="Previously I sent you a picture of an opened fridge, whare are there in the fridge, do you remember?"
)

response = agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Tell me what are there in the fridge'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='0ab31721-370c-4f38-ad59-850afc41655b'),
              AIMessage(content='Here’s what I can see in the fridge, from top to bottom:\n\n- Top shelf\n  - Olive oil bottle\n  - Half a lime\n  - Cucumber\n  - Celery stalks\n  - Green apple\n\n- Middle shelf\n  - Beet greens (with red stems)\n  - Strawberries\n  - Beet (root)\n  - Bottle of red juice (likely tomato juice)\n\n- Bottom shelf\n  - Carrots with greens\n  - Two bottles of orange juice\n  - Citrus: one orange cut in half, plus two whole oranges\n  - Two lemons\n\nWant me to suggest a quick snack or a recipe using these ingredients?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1361, 'prompt_tokens': 1559, 'total_tokens': 2920, 'completion_tokens_details': {'accepted_predicti

### Use <span style="color: gold;">web search tool</span> & <span style="color: gold;">memory</span>

In [ ]:
question = HumanMessage(
    content="Please use web search tool, think of what dish you can make out of the ingredients inside of the fridge."
)

response = agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Tell me what are there in the fridge'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='0ab31721-370c-4f38-ad59-850afc41655b'),
              AIMessage(content='Here’s what I can see in the fridge, from top to bottom:\n\n- Top shelf\n  - Olive oil bottle\n  - Half a lime\n  - Cucumber\n  - Celery stalks\n  - Green apple\n\n- Middle shelf\n  - Beet greens (with red stems)\n  - Strawberries\n  - Beet (root)\n  - Bottle of red juice (likely tomato juice)\n\n- Bottom shelf\n  - Carrots with greens\n  - Two bottles of orange juice\n  - Citrus: one orange cut in half, plus two whole oranges\n  - Two lemons\n\nWant me to suggest a quick snack or a recipe using these ingredients?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1361, 'prompt_tokens': 1559, 'total_tokens': 2920, 'completion_tokens_details': {'accepted_predicti

In [23]:
print(response["messages"][-1].content)

Nice—those ingredients work well for a fresh beet & carrot citrus salad with olive oil dressing. Here’s a dish you can make using what’s in your fridge, plus a couple of similar ideas.

Primary dish: Beet & Carrot Citrus Salad with Olive Oil Dressing
- What to use from your fridge:
  - Beets (root) – roasted or boiled, peeled and shaved into ribbons
  - Beet greens can be lightly sautéed or used as a bed
  - Carrots – shaved into ribbons or grated
  - Oranges or lemons (juice and/or segments) for the citrus dressing
  - Olive oil
  - Optional: strawberries for color, a little apple for sweetness, or celery for crunch

- Simple dressing (olive oil citrus vinaigrette):
  - 3 parts olive oil to 1 part citrus juice (use lemon and/or orange)
  - A pinch of salt and pepper
  - Optional additions if you have them: a touch of Dijon mustard or honey for balance
  - Whisk together and toss with the shaved beets, carrots, and citrus segments

- Steps:
  1) Cook the beets until tender, cool, peel,